# NLP Ticket Classification - Data Exploration

This notebook explores the ticket dataset and analyzes the text data before training the BERT model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load the dataset
df = pd.read_csv('../data/tickets.csv')
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
df.head()

In [ ]:
# Check for missing values
print("Missing values:")
df.isnull().sum()

In [ ]:
# Analyze label distribution
label_counts = df['label'].value_counts()
print("Label distribution:")
print(label_counts)

# Plot label distribution
plt.figure(figsize=(10, 6))
label_counts.plot(kind='bar')
plt.title('Distribution of Ticket Categories')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze text length distribution
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("Text statistics:")
print(df[['text_length', 'word_count']].describe())

# Plot text length distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.hist(df['text_length'], bins=30, alpha=0.7)
ax1.set_title('Distribution of Text Length (characters)')
ax1.set_xlabel('Text Length')
ax1.set_ylabel('Frequency')

ax2.hist(df['word_count'], bins=30, alpha=0.7, color='orange')
ax2.set_title('Distribution of Word Count')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Text analysis by category
def analyze_text_by_category(df, category_col='label', text_col='text'):
    """Analyze text characteristics by category"""
    analysis = {}
    
    for category in df[category_col].unique():
        category_texts = df[df[category_col] == category][text_col]
        
        analysis[category] = {
            'count': len(category_texts),
            'avg_length': category_texts.str.len().mean(),
            'avg_words': category_texts.str.split().str.len().mean(),
            'sample_text': category_texts.iloc[0] if len(category_texts) > 0 else ""
        }
    
    return pd.DataFrame(analysis).T

category_analysis = analyze_text_by_category(df)
print("Analysis by category:")
category_analysis

In [ ]:
# Word frequency analysis
def get_word_frequency(texts, top_n=20):
    """Get most common words"""
    all_words = []
    for text in texts:
        # Simple preprocessing
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        all_words.extend(words)
    
    return Counter(all_words).most_common(top_n)

# Overall word frequency
word_freq = get_word_frequency(df['text'])
print("Top 20 most common words:")
for word, count in word_freq:
    print(f"{word}: {count}")

In [ ]:
# Word frequency by category
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, category in enumerate(df['label'].unique()):
    if i < len(axes):
        category_texts = df[df['label'] == category]['text']
        word_freq_cat = get_word_frequency(category_texts, top_n=10)
        
        words, counts = zip(*word_freq_cat)
        
        axes[i].bar(words, counts)
        axes[i].set_title(f'Category {category} - Top Words')
        axes[i].set_xlabel('Words')
        axes[i].set_ylabel('Frequency')
        axes[i].tick_params(axis='x', rotation=45)

# Hide any unused subplots
for i in range(len(df['label'].unique()), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

## Key Findings

1. **Dataset Overview**: The dataset contains X tickets across Y categories
2. **Class Distribution**: [Describe if balanced or imbalanced]
3. **Text Characteristics**: Average text length and word count
4. **Common Patterns**: Most frequent words and category-specific terms

## Next Steps

1. Data preprocessing and cleaning
2. Text tokenization for BERT
3. Train-test split preparation
4. Model training and evaluation